# Network Flow V2 — ToN-IoT Network: Step 01 / Dataset Audit

**Exact local source:** `data/raw/ton_iot/ton_iot_train_test_network.csv`. This notebook reads only; it never changes the raw CSV.

**Purpose:** list actual column names and sampled types; read the whole file **in bounded chunks**; count labels, missing cells and rows; calculate a SHA-256 checksum; save an audit JSON report.

**Important:** The common network split generally supplies binary label and multiclass type. This notebook verifies rather than assumes their presence or semantics. Values reported for other versions of this dataset are not a substitute for counts from your file. No train/test splitting occurs here.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from IPython.display import display
import hashlib
import json
import pandas as pd

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
DATA = ROOT / "data" / "raw" / "ton_iot" / 'ton_iot_train_test_network.csv'
REPORT_DIR = ROOT / "reports" / "audits"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = REPORT_DIR / "ton_iot_audit_v1.json"
CHUNK_SIZE = 25000

print("Source:", DATA)
print("Exists:", DATA.is_file())
if not DATA.is_file():
    raise FileNotFoundError(f"Check your dataset path: {DATA}")
print(f"File size: {DATA.stat().st_size / 1024**2:,.2f} MB")


## 1. Sample the schema

Sampled types are **tentative**: numeric-looking strings, timestamps and mixed-type columns may be inferred differently later. Do not treat a column name as proof of physical units or flow semantics.

In [ ]:
sample = pd.read_csv(DATA, nrows=1000, low_memory=False)
COLUMNS = sample.columns.tolist()
if len(set(COLUMNS)) != len(COLUMNS):
    raise ValueError("Duplicated column names: investigate the CSV header before proceeding")
print(f"Columns: {len(COLUMNS)}")
display(pd.DataFrame({"column": COLUMNS, "sampled_dtype": [str(sample[c].dtype) for c in COLUMNS]}))
print("Sample records inspected:", len(sample))
# Do not print raw rows: datasets can contain IPs, URLs, credentials or other sensitive payloads.


## 2. Identify **candidate** label columns

These are names worth investigating, not automatically verified ground truth. Check semantics in the release documentation before model training.

In [ ]:
def normalized_name(value):
    return "".join(ch for ch in str(value).casefold() if ch.isalnum())

LABEL_NAMES = set(['attacklabel', 'attacktype', 'label', 'type', 'class1', 'class2', 'class3', 'attacksubtype', 'attackcategory', 'target', 'maliciouslabel'])
LABEL_COLUMNS = [c for c in COLUMNS if normalized_name(c) in LABEL_NAMES]
FEATURE_COLUMNS = [c for c in COLUMNS if c not in LABEL_COLUMNS]
print("Candidate label columns:", LABEL_COLUMNS)
print("Candidate non-label columns:", len(FEATURE_COLUMNS))
if not LABEL_COLUMNS:
    print("WARNING: No target column recognized. Share the column names for manual label mapping.")


## 3. Count **every record** using chunks

This counts all CSV rows, `pandas`-recognized nulls, and the distribution for each candidate label. Raw values of non-label fields are not saved. Large files are not loaded in full.

In [ ]:
row_count = 0
null_counts = Counter()
label_counts = {c: Counter() for c in LABEL_COLUMNS}

for n, chunk in enumerate(pd.read_csv(DATA, chunksize=CHUNK_SIZE, dtype="string", low_memory=False), start=1):
    row_count += len(chunk)
    null_counts.update({c: int(v) for c, v in chunk.isna().sum().items()})
    for label in LABEL_COLUMNS:
        counts = chunk[label].fillna("<MISSING>").str.strip().value_counts(dropna=False)
        label_counts[label].update({str(name): int(count) for name, count in counts.items()})
    if n % 10 == 0:
        print(f"Chunks read: {n}; total rows: {row_count:,}")

print(f"FULL COUNT: {row_count:,} records, {len(COLUMNS)} columns")
print("Top null columns:")
display(pd.DataFrame([{"feature": c, "null_count": null_counts[c],
                      "null_pct": round(null_counts[c] * 100 / max(row_count, 1), 4)}
                     for c in COLUMNS]).sort_values("null_count", ascending=False).head(20))


## 4. Label distribution

Compare candidate label levels before deciding which is binary (Normal/Attack) and which gives attack category. Do not substitute labels for input features.

In [ ]:
if not LABEL_COLUMNS:
    print("No candidate labels detected; manual mapping required.")
else:
    for label in LABEL_COLUMNS:
        pairs = label_counts[label].most_common()
        print(f"\nLabel column: {label} — distinct values: {len(pairs):,}")
        table = pd.DataFrame([{"label": name, "count": count,
                               "percentage": round(count * 100 / max(row_count, 1), 4)}
                              for name, count in pairs])
        display(table.head(50))
        if len(table) > 50:
            print("Only the top 50 are shown; full counts are saved in JSON.")


## 5. Save the SHA-256-identified audit report

The checksum helps detect accidental changes to the input file. This report contains aggregate counts, not raw records.

In [ ]:
digest = hashlib.sha256()
with DATA.open("rb") as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
        digest.update(block)

report = {
    "audit_version": "v1",
    "dataset": 'ToN-IoT Network',
    "filename": DATA.name,
    "file_size_bytes": DATA.stat().st_size,
    "sha256": digest.hexdigest(),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "row_count": row_count,
    "column_count": len(COLUMNS),
    "column_names": COLUMNS,
    "sample_dtypes": {c: str(sample[c].dtype) for c in COLUMNS},
    "candidate_label_columns": LABEL_COLUMNS,
    "candidate_feature_columns": FEATURE_COLUMNS,
    "null_counts": dict(null_counts),
    "candidate_label_counts": {c: dict(counts) for c, counts in label_counts.items()},
    "note": "Candidate labels are not automatically verified; sampled types are not final.",
}
with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print("Saved:", REPORT_PATH)
print("SHA-256:", report["sha256"])


## What to share

Send **total rows, column names, candidate label columns, label distributions, and top missing-value columns**. Then run the matching **Step 02 Feature Quality** notebook. Do not split or normalize yet.